## Performance

I'm going to run crossmatches between ZTF and Gaia and have a look at the metrics.

In [1]:
import dask
import json
import lsdb
import ray
import plotly.graph_objects as go
import tempfile
import time

from lsdb import ConeSearch
from dask.distributed import Client
from distributed.diagnostics.memory_sampler import MemorySampler
from plotly.subplots import make_subplots
from ray_mem_sampler import RayMemorySampler
from ray.util.dask import ray_dask_get

In [2]:
def get_crossmatch(radius_deg):
    """Run crossmatch between ZTF and Gaia catalogs for a specified radius"""
    ztf = lsdb.open_catalog(
        "/mnt/data/hats/catalogs/ztf_dr22",
        columns=["objectid", "objra", "objdec"],
        search_filter=ConeSearch(ra=270, dec=20, radius_arcsec=radius_deg*3600),
    )
    gaia = lsdb.open_catalog(
        "/mnt/data/hats/catalogs/v06/gaia_dr3",
        columns=["source_id", "ra", "dec"],
        search_filter=ConeSearch(ra=270, dec=20, radius_arcsec=radius_deg*3600),
    )
    return ztf.crossmatch(gaia)

Example of running a crossmatch with dask-on-ray:

```python
ray.init(num_cpus=4)

catalog = get_crossmatch(10)

# Using enable_dask_on_ray context manager we get some warnings
# about deprecations of Dask configurations. 
# E.g. shuffle optimization is not supported since dask 2025.1.0.
# To avoid that we set the scheduler directly to ray_dask_get and
# do not set the shuffle or dataframe optimizations.
with dask.config.set(scheduler=ray_dask_get):
    catalog.compute()

ray.shutdown()
```

Let's compare the crossmatch performance with the increasing radius size:

In [3]:
def _run_action(catalog, write: bool) -> None:
    """Compute or write the catalog depending on the write flag."""
    if write:
        with tempfile.TemporaryDirectory() as tmp:
            catalog.write_catalog(tmp)
    else:
        catalog.compute()


def run_dask(catalog, write: bool = False) -> dict:
    """Run with the native Dask distributed scheduler."""
    ms = MemorySampler()
    t0 = time.time()
    with Client(n_workers=4, memory_limit=None):
        with ms.sample("dask"):
            _run_action(catalog, write)
    elapsed = time.time() - t0
    peak_gb = ms.to_pandas()["dask"].max() / 2**30
    print(f"  [dask] {elapsed:.1f}s | peak mem {peak_gb:.2f} GiB")
    return {"dask_time": elapsed, "dask_peak_gb": peak_gb}


def run_ray(catalog, write: bool = False) -> dict:
    """Run with the Dask-on-Ray scheduler."""
    ray.init(num_cpus=4, ignore_reinit_error=True)
    with RayMemorySampler() as ms:
        t0 = time.time()
        with dask.config.set(scheduler=ray_dask_get):
            _run_action(catalog, write)
        elapsed = time.time() - t0
    ray.shutdown()
    print(
        f"  [ray]  {elapsed:.1f}s"
        f" | peak mem {ms.peak_cluster_mem_gb:.2f} GiB"
        f" | object store {ms.peak_object_store_gb:.2f} GiB"
    )
    return {
        "ray_time": elapsed,
        "ray_peak_cluster_gb": ms.peak_cluster_mem_gb,
        "ray_peak_object_store_gb": ms.peak_object_store_gb,
    }

def benchmark_with_deg(radius_deg: float, write: bool = False) -> dict:
    """Run crossmatch with both schedulers and collect runtime + peak memory."""
    catalog = get_crossmatch(radius_deg)
    n_partitions = len(catalog.get_healpix_pixels())

    print(f"Radius: {radius_deg} deg  |  Partitions: {n_partitions}  |  write={write}")
    dask_metrics = run_dask(catalog, write)
    ray_metrics = run_ray(catalog, write)

    return {
        "radius_deg": radius_deg,
        "n_partitions": n_partitions,
        **dask_metrics,
        **ray_metrics,
    }

In [4]:
results_compute = [benchmark_with_deg(deg, write=False) for deg in [1, 5, 10]]
with open("results/results_compute.json", "w") as f:
    json.dump(results_compute, f, indent=2)
results_compute

/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/lsdb/catalog/catalog.py:358: FutureWarning: The default suffix behavior will change from applying suffixes to all columns to only applying suffixes to overlapping columns in a future release.To maintain the current behavior, explicitly set `suffix_method='all_columns'`. To change to the new behavior, set `suffix_method='overlapping_columns'`.
  warnings.warn(


Radius: 1 deg  |  Partitions: 4  |  write=False
  [dask] 8.9s | peak mem 1.61 GiB


2026-04-10 13:56:19,874	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 
/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/ray/_private/worker.py:2052: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Computing Catalog:   0%|          | 0/83 [00:00<?, ?it/s]

  [ray]  6.2s | peak mem 1.12 GiB | object store 0.00 GiB
Radius: 5 deg  |  Partitions: 33  |  write=False


...

  [dask] 58.6s | peak mem 14.25 GiB


2026-04-10 13:58:19,540	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


Computing Catalog:   0%|          | 0/1691 [00:00<?, ?it/s]

  [ray]  59.0s | peak mem 15.22 GiB | object store 2.79 GiB


[{'radius_deg': 1,
  'n_partitions': 4,
  'dask_time': 8.899879455566406,
  'dask_peak_gb': np.float64(1.6126670837402344),
  'ray_time': 6.237532377243042,
  'ray_peak_cluster_gb': 1.115368,
  'ray_peak_object_store_gb': 0.0},
 {'radius_deg': 5,
  'n_partitions': 33,
  'dask_time': 18.06223440170288,
  'dask_peak_gb': np.float64(4.123638153076172),
  'ray_time': 16.31731104850769,
  'ray_peak_cluster_gb': 7.73212,
  'ray_peak_object_store_gb': 1.5129817361012101},
 {'radius_deg': 10,
  'n_partitions': 161,
  'dask_time': 58.603362798690796,
  'dask_peak_gb': np.float64(14.247974395751953),
  'ray_time': 59.028770208358765,
  'ray_peak_cluster_gb': 15.224488000000001,
  'ray_peak_object_store_gb': 2.79440494813025}]

In [5]:
results_write = [benchmark_with_deg(deg, write=True) for deg in [1, 5, 10]]
with open("results/results_write.json", "w") as f:
    json.dump(results_write, f, indent=2)
results_write

Radius: 1 deg  |  Partitions: 4  |  write=True


/home/scampos/notebooks_lf/lsdb/performance_week_2026/4.dask-on-ray/.venv/lib/python3.12/site-packages/lsdb/catalog/catalog.py:358: FutureWarning: The default suffix behavior will change from applying suffixes to all columns to only applying suffixes to overlapping columns in a future release.To maintain the current behavior, explicitly set `suffix_method='all_columns'`. To change to the new behavior, set `suffix_method='overlapping_columns'`.
  warnings.warn(


  [dask] 8.4s | peak mem 1.56 GiB


2026-04-10 13:59:37,459	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


Writing Catalog:   0%|          | 0/44 [00:00<?, ?it/s]

...

  [dask] 97.5s | peak mem 3.36 GiB


2026-04-10 14:02:44,039	INFO worker.py:2004 -- Started a local Ray instance. View the dashboard at http://127.0.0.1:8265 


Writing Catalog:   0%|          | 0/1211 [00:00<?, ?it/s]

  [ray]  133.9s | peak mem 9.16 GiB | object store 1.05 GiB


[{'radius_deg': 1,
  'n_partitions': 4,
  'dask_time': 8.374775648117065,
  'dask_peak_gb': np.float64(1.5647735595703125),
  'ray_time': 7.109641075134277,
  'ray_peak_cluster_gb': 1.111144,
  'ray_peak_object_store_gb': 0.0},
 {'radius_deg': 5,
  'n_partitions': 33,
  'dask_time': 26.8713321685791,
  'dask_peak_gb': np.float64(3.060291290283203),
  'ray_time': 33.25778079032898,
  'ray_peak_cluster_gb': 5.897844,
  'ray_peak_object_store_gb': 0.5890784952789545},
 {'radius_deg': 10,
  'n_partitions': 161,
  'dask_time': 97.51668071746826,
  'dask_peak_gb': np.float64(3.357086181640625),
  'ray_time': 133.94788718223572,
  'ray_peak_cluster_gb': 9.163372,
  'ray_peak_object_store_gb': 1.0502728261053562}]

In [9]:
def plot_metrics(results):
    """Plot runtime and memory metrics for Dask vs Dask-on-Ray."""
    labels = [f"{r['radius_deg']} deg ({r['n_partitions']} parts)" for r in results]

    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=(
            "Total runtime (s)",
            "Peak cluster memory (GiB)",
            "Peak object store memory (GiB)",
        ),
    )

    common = dict(x=labels, width=0.3)
    colors = {"dask": "#1f77b4", "ray": "#ff7f0e"}

    # Runtime
    fig.add_trace(
        go.Bar(name="Native Dask", y=[r["dask_time"] for r in results],
            marker_color=colors["dask"], legendgroup="dask", **common),
        row=1, col=1,
    )
    fig.add_trace(
        go.Bar(name="Dask-on-Ray", y=[r["ray_time"] for r in results],
            marker_color=colors["ray"], legendgroup="ray", **common),
        row=1, col=1,
    )

    # Peak memory
    fig.add_trace(
        go.Bar(name="Native Dask", y=[r["dask_peak_gb"] for r in results],
            marker_color=colors["dask"], legendgroup="dask", showlegend=False, **common),
        row=1, col=2,
    )
    fig.add_trace(
        go.Bar(name="Dask-on-Ray", y=[r["ray_peak_cluster_gb"] for r in results],
            marker_color=colors["ray"], legendgroup="ray", showlegend=False, **common),
        row=1, col=2,
    )

    # Dask-on-ray object store memory
    fig.add_trace(
        go.Bar(name="Dask-on-Ray", y=[r["ray_peak_object_store_gb"] for r in results],
            marker_color=colors["ray"], legendgroup="ray", showlegend=False, **common),
        row=1, col=3,
    )

    fig.update_layout(
        title="ZTF x Gaia crossmatch: Dask vs Dask-on-Ray (4 workers)",
        barmode="group",
        legend=dict(orientation="h", yanchor="bottom", y=1.08, xanchor="center", x=0.5),
        height=450,
        template="plotly_white",
    )
    fig.update_yaxes(title_text="seconds", row=1, col=1)
    fig.update_yaxes(title_text="GiB", row=1, col=2)
    fig.update_yaxes(title_text="GiB", row=1, col=3)
    fig.show()

In [ ]:
plot_metrics(results_compute)

Wait expired, Browser is being closed by watchdog.
Wait expired, Browser is being closed by watchdog.


In [11]:
plot_metrics(results_write)

### Ray Dask Callbacks

We can use callbacks within a ray context to get more information about the Ray tasks lifecycles.

In [12]:
from ray.util.dask import RayDaskCallback

class Callbacks(RayDaskCallback):
    """All ray callbacks"""

    def _ray_presubmit(self, task, key, deps):
        # Before submitting a task
        print(f"About to submit task {key}!")

    def ray_postsubmit(self, task, key, deps, object_ref):
        # After submitting a task
        print(f"Submitted task {key}!")

    def ray_pretask(key, object_refs):
        # Before executing a task, inside Ray worker
        print(f"About to execute task {key}!")

    def ray_posttask(key, result, pre_state):
        # After executing a task, inside Ray worker
        print(f"Finished executing task {key}!")

    def ray_postsubmit_all(self, object_refs, dsk):
        # Run after all Ray tasks are submitted
        print("All tasks submitted!")

    def ray_finish(self, result):
        # Run after all tasks are finished
        print("All tasks finished!")

In [ ]:
with dask.config.set(scheduler=ray_dask_get), Callbacks():
    benchmark_with_deg(1)

### Takeaways

A major difference between Dask and Ray is how memory is handled:

|  | Dask `memory_limit` | Ray `_memory` |
|---|---|---|
| Scope | Per worker | Total node budget |
| Enforced at the OS-level?| Yes | No |
| OOM behavior | Spill to disk / restart worker | No action |
| Used for scheduling? | Yes | Yes |

We can set memory requirements for tasks, but again, they are only enforced in scheduling.

Using dask-on-ray on shared machines (e.g. arnor) is therefore **not advisable**, unless in a containerized environment.